In [2]:
import moabb
from moabb.datasets import *
from moabb.evaluations import CrossSubjectEvaluation
from moabb.paradigms import P300
from hoda.hoda import HODA
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.analysis.plotting import paired_plot, meta_analysis_plot, summary_plot
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA
from sklearn.svm import SVC
from moabb.analysis.meta_analysis import (  # noqa: E501
    compute_dataset_statistics,
    find_significant_differences,
)

In [3]:
tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = fmax*2

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #bi2013a(),
    #bi2014a(),
    #bi2014b(),
    #bi2015a(),
    #bi2015b(),
    BNCI2014008(),
    #BNCI2014009(),
    #BNCI2015003(),
    #DemonsP300(),
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019()

]
evaluation = CrossSubjectEvaluation(
    paradigm=paradigm, datasets=datasets,
    suffix="cross_session", overwrite=True,
)

BNCI2014008 has been renamed to BNCI2014_008. BNCI2014008 will be removed in version 0.7.
The dataset class name 'BNCI2014008' must be an abbreviation of its code 'BNCI2014-008'. See moabb.datasets.base.is_abbrev for more information.


In [4]:
for dataset in datasets:
    dataset.download()

In [5]:
import numpy as np

class ToeplitzLDAWrapper(BaseEstimator, ClassifierMixin):

    def fit(self, X, y=None):
        n_epochs, n_channels, n_samples = X.shape
        self.tlda_ = ToeplitzLDA(n_channels=n_channels, data_is_channel_prime=False)
        X = X.reshape(n_epochs, -1)
        return self.tlda_.fit(X, y)

    def decision_function(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.decision_function(X)

    def predict(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict(X)

    def predict_proba(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict_proba(X)



def lagged_tensor(X, y=None):    
    window = 0.2
    max_lag = 0.5
    
    n_epochs, n_channels, _ = X.shape
    roi0 = int((0 - tmin)*sfreq)
    roi1 = int((window - tmin)*sfreq)
    n_times = roi1-roi0
    n_lags= int(max_lag*sfreq)
    Xt = np.zeros((n_epochs, n_channels, n_times, n_lags))
    for l in range(n_lags):
        Xt[:,:,:,l] = X[:,:,roi0+l:roi1+l]
    return Xt

def reshape(X, y=None):
    return X.reshape((X.shape[0],-1))

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import SVC
from sklearn.feature_selection import SelectPercentile, f_classif
from mne.decoding import Scaler
from hoda.hoda import HODA, BTTDA
import cupy

pipelines = dict()



pipelines['HODA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=32,
        rank=None,
        tol=1e-6,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='rt',
        solver='lanczos',        
        verbose=False,
        taper=False,
        lasso=False,
        prune=True,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(),
)


pipelines['BTTDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=2,
        hoda_params=dict(
            max_iter=32,
            rank=None,
            tol=1e-6,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            keep_train_info=False
        ),
    ),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['tLDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    ToeplitzLDAWrapper()
)

In [10]:
import tensorly as tl
tl.set_backend('cupy', local_threadsafe=True)

In [11]:
results = evaluation.process(pipelines)

BNCI2014-008-CrossSubject:   0%|                                                                | 0/8 [00:00<?, ?it/s]/data/.virtualenvs/hoda/lib64/python3.11/site-packages/scipy/stats/_stats_py.py:9690: RuntimeWarning: divide by zero encountered in log
  statistic = -2 * np.sum(np.log(pvalues))


Fitted tucker model of rank [8, 7] ...


BNCI2014-008-CrossSubject:   0%|                                                                | 0/8 [00:04<?, ?it/s]

Fitted tucker model of rank [4, 5] ...


TypeError: Implicit conversion to a NumPy array is not allowed. Please use `.get()` to construct a NumPy array explicitly.

In [ ]:
results

In [ ]:
stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
_=meta_analysis_plot(stats, "tLDA", "lagHODA")

In [ ]:
_=paired_plot(results, "tLDA", "lagHODA")